# Tragedy – "Epicness" over Time

In [4]:
import altair as alt
import numpy as np
import pandas as pd

homer_df = pd.read_parquet("../parquet/homer_dtm.parquet")
tragedy_df = pd.read_parquet("../parquet/tragedy_dtm.parquet")
loglikelihood_df = pd.read_parquet("../parquet/epic_tragic_lemmata_loglikelihood.parquet")

play_year = (
    pd.read_parquet("../parquet/tragedy-with-years.parquet")
    .groupby(["dramatist", "title"])["year"]
    .first()
)

ll = loglikelihood_df.set_index("lemma")["log_likelihood"]
speaker_avg = tragedy_df.multiply(ll, axis=0).sum() / tragedy_df.sum()
speaker_df = speaker_avg.reset_index(name="avg_log_likelihood")
speaker_df["year"] = speaker_df.set_index(["dramatist", "title"]).index.map(play_year)

alt.Chart(speaker_df).mark_line(point=True).encode(
    x=alt.X("year:O", title="Year"),
    y=alt.Y("mean(avg_log_likelihood):Q", title="Avg log-likelihood (negative = more 'tragic')"),
    color="dramatist:N",
)

alt.Chart(...)

In [13]:
alt.Chart(speaker_df).mark_point().encode(
    x=alt.X("year:Q", title="Year", scale=alt.Scale(domain=[-480, -401])),
    y=alt.Y("avg_log_likelihood:Q", title="Avg log-likelihood"),
    color="dramatist:N",
) + alt.Chart(speaker_df).mark_line(color="black", strokeDash=[4, 4]).encode(
    x="year:Q",
    y="avg_log_likelihood:Q",
).transform_regression("year", "avg_log_likelihood")

alt.LayerChart(...)

In [12]:
alt.Chart(speaker_df).mark_point().encode(
    x=alt.X("year:Q", title="Year", scale=alt.Scale(domain=[-480, -401])),
    y=alt.Y("avg_log_likelihood:Q"),
    color="dramatist:N",
) + alt.Chart(speaker_df).mark_line().encode(
    x="year:Q",
    y="avg_log_likelihood:Q",
    color="dramatist:N",
    strokeDash="dramatist:N",
).transform_regression("year", "avg_log_likelihood", groupby=["dramatist"])

alt.LayerChart(...)

In [14]:
from scipy import stats

slope, intercept, r, p, se = stats.linregress(
    speaker_df["year"], speaker_df["avg_log_likelihood"]
)
print(f"slope={slope:.4f}, r={r:.4f}, p={p:.6f}")

for dramatist, grp in speaker_df.groupby("dramatist"):
    slope, intercept, r, p, se = stats.linregress(grp["year"], grp["avg_log_likelihood"])
    print(f"{dramatist}: slope={slope:.4f}, p={p:.6f}")

slope=-0.2193, r=-0.1568, p=0.006329
Aeschylus: slope=0.3218, p=0.547604
Euripides: slope=0.1486, p=0.385399
Sophocles: slope=-0.4429, p=0.044868
